# Filtrar_Semanal_Stoplist

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Filtra una lista de stopwords (stop_list_español.txt) de todos los archivos .txt
en las carpetas:

    Youtube_Semanal_Txt
    X_Twitter_Semanal_Txt
    Fuentes_Integradas_Txt
    Facebook_Semanal_Txt

ubicadas en:
    /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/

Crea 4 carpetas nuevas en el mismo directorio:

    Youtube_Semanal_Txt_Filtrado
    X_Twitter_Semanal_Txt_Filtrado
    Fuentes_Integradas_Txt_Filtrado
    Facebook_Semanal_Txt_Filtrado

y en cada una guarda los archivos filtrados con el mismo nombre que el original
pero agregando el sufijo "_Filtrado" ANTES de la fecha final:

Ejemplo:
    X_Twitter_Todo_Ordenado_Homologados_Semanas_2025-11-11.txt
    -> X_Twitter_Todo_Ordenado_Homologados_Semanas_Filtrado_2025-11-11.txt
"""

from pathlib import Path
import re

# Rutas base
BASE_DIR = Path(
    "/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/"
)

STOPLIST_PATH = BASE_DIR / "Diccionarios" / "stop_list_español.txt"

CARPETAS_ORIGINALES = [
    "Youtube_Semanal_Txt",
    "X_Twitter_Semanal_Txt",
    "Fuentes_Integradas_Txt",
    "Facebook_Semanal_Txt",
]


def cargar_stopwords(ruta_stoplist: Path) -> set:
    """
    Carga la lista de palabras vacías (una por línea).
    Devuelve un set en minúsculas para comparación rápida.
    """
    stopwords = set()
    with ruta_stoplist.open("r", encoding="utf-8") as f:
        for linea in f:
            palabra = linea.strip()
            if palabra:
                stopwords.add(palabra.lower())
    return stopwords


def filtrar_linea(linea: str, stopwords: set) -> str:
    """
    Filtra de una línea todas las palabras que estén en el conjunto stopwords.
    - Comparación en minúsculas.
    - Se conserva el salto de línea (pero se colapsan espacios múltiples).
    """
    tokens = linea.strip().split()
    tokens_filtrados = [t for t in tokens if t.lower() not in stopwords]
    return " ".join(tokens_filtrados)


def nombre_salida_con_fecha(archivo: Path) -> str:
    """
    Inserta '_Filtrado' antes de la fecha final (YYYY-MM-DD) si existe.
    Si no se detecta fecha, añade '_Filtrados' al final del nombre.
    """
    stem = archivo.stem
    # Busca patrón con fecha al final: *_YYYY-MM-DD
    m = re.match(r"^(.*)_([0-9]{4}-[0-9]{2}-[0-9]{2})$", stem)
    if m:
        base, fecha = m.groups()
        nuevo_stem = f"{base}_Filtrado_{fecha}"
    else:
        nuevo_stem = f"{stem}_Filtrados"
    return nuevo_stem + archivo.suffix


def procesar_carpeta(nombre_carpeta: str, stopwords: set) -> None:
    """
    Procesa todos los .txt de una carpeta origen y los deja filtrados
    en la carpeta destino con sufijo _Filtrado.
    """
    carpeta_origen = BASE_DIR / nombre_carpeta
    carpeta_destino = BASE_DIR / f"{nombre_carpeta}_Filtrado"
    carpeta_destino.mkdir(parents=True, exist_ok=True)

    print(f"\nProcesando carpeta: {carpeta_origen}")
    print(f"Carpeta destino   : {carpeta_destino}")

    if not carpeta_origen.exists():
        print(f"  [ADVERTENCIA] No existe la carpeta origen: {carpeta_origen}")
        return

    for archivo in sorted(carpeta_origen.iterdir()):
        if not archivo.is_file():
            continue
        if archivo.suffix.lower() != ".txt":
            continue

        ruta_salida = carpeta_destino / nombre_salida_con_fecha(archivo)

        with archivo.open("r", encoding="utf-8") as f_in, \
             ruta_salida.open("w", encoding="utf-8") as f_out:

            for linea in f_in:
                linea_filtrada = filtrar_linea(linea, stopwords)
                f_out.write(linea_filtrada + "\n")

        print(f"  Archivo procesado: {archivo.name} -> {ruta_salida.name}")


def main():
    if not STOPLIST_PATH.exists():
        raise FileNotFoundError(f"No encuentro la stoplist: {STOPLIST_PATH}")

    print(f"Usando stoplist: {STOPLIST_PATH}")
    stopwords = cargar_stopwords(STOPLIST_PATH)
    print(f"Stopwords cargadas: {len(stopwords)} palabras\n")

    for nombre_carpeta in CARPETAS_ORIGINALES:
        procesar_carpeta(nombre_carpeta, stopwords)

    print("\n--- Proceso terminado ---")


if __name__ == "__main__":
    main()


Usando stoplist: /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios/stop_list_español.txt
Stopwords cargadas: 758 palabras


Procesando carpeta: /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Youtube_Semanal_Txt
Carpeta destino   : /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Youtube_Semanal_Txt_Filtrado
  Archivo procesado: Youtube_Todo_Ordenado_Homologados_Semanas_2024-10-29.txt -> Youtube_Todo_Ordenado_Homologados_Semanas_Filtrado_2024-10-29.txt
  Archivo procesado: Youtube_Todo_Ordenado_Homologados_Semanas_2024-11-05.txt -> Youtube_Todo_Ordenado_Homologados_Semanas_Filtrado_2024-11-05.txt
  Archivo procesado: Youtube_Todo_Ordenado_Homologados_Semanas_2024-11-12.txt -> Youtube_Todo_Ordenado_Homologados_Semanas_Filtrado_2024-11-12.txt
  Archivo procesado:

# Clasifficacion_Polaridad 

In [8]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import re
from datetime import datetime
import numpy as np

# Rutas base
base_path = "/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido"
resultados_dir = os.path.join(base_path, "Resultados_Clasificacion_Semanal_Etiquetado_Filtrado")

# Carpetas a analizar
carpetas_analizar = [
    "Facebook_Semanal_Txt_Filtrado",
    "Fuentes_Integradas_Txt_Filtrado", 
    "X_Twitter_Semanal_Txt_Filtrado",
    "Youtube_Semanal_Txt_Filtrado"
]

# Rutas de diccionarios - usando los nuevos diccionarios con n-gramas
diccionarios_base = "/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios"
ruta_positivas = os.path.join(diccionarios_base, "diccionario_positivo_etiquetado.txt")
ruta_negativas = os.path.join(diccionarios_base, "diccionario_negativo_etiquetado.txt")
ruta_seguridad = os.path.join(diccionarios_base, "diccionario_seguridad.txt")
ruta_inseguridad = os.path.join(diccionarios_base, "diccionario_inseguridad.txt")

# Crear carpeta de resultados si no existe
if not os.path.exists(resultados_dir):
    os.makedirs(resultados_dir)
    print(f"Carpeta de resultados creada: {resultados_dir}")

# Función simple para leer un archivo de texto
def leer_archivo(ruta):
    try:
        with open(ruta, 'r', encoding='utf-8') as f:
            return f.read()
    except:
        try:
            with open(ruta, 'r', encoding='latin-1') as f:
                return f.read()
        except Exception as e:
            print(f"Error al leer {ruta}: {e}")
            return ""

# Función para separar palabras individuales de n-gramas
def cargar_diccionario_con_ngramas(ruta):
    texto = leer_archivo(ruta)
    palabras_individuales = set()
    ngramas = set()
    
    for linea in texto.splitlines():
        termino = linea.strip().lower()
        if termino:
            if ' ' in termino:  # Es un n-grama (tiene espacios)
                ngramas.add(termino)
            else:  # Es una palabra individual
                palabras_individuales.add(termino)
    
    return palabras_individuales, ngramas

# Cargar palabras positivas y n-gramas positivos
print(f"Leyendo palabras/expresiones positivas de {ruta_positivas}")
palabras_positivas, ngramas_positivos = cargar_diccionario_con_ngramas(ruta_positivas)
print(f"Palabras positivas cargadas: {len(palabras_positivas)}")
print(f"N-gramas positivos cargados: {len(ngramas_positivos)}")

# Cargar palabras negativas y n-gramas negativos
print(f"Leyendo palabras/expresiones negativas de {ruta_negativas}")
palabras_negativas, ngramas_negativos = cargar_diccionario_con_ngramas(ruta_negativas)
print(f"Palabras negativas cargadas: {len(palabras_negativas)}")
print(f"N-gramas negativos cargados: {len(ngramas_negativos)}")

# Cargar palabras de seguridad y n-gramas de seguridad
print(f"Leyendo palabras/expresiones de seguridad de {ruta_seguridad}")
palabras_seguridad, ngramas_seguridad = cargar_diccionario_con_ngramas(ruta_seguridad)
print(f"Palabras de seguridad cargadas: {len(palabras_seguridad)}")
print(f"N-gramas de seguridad cargados: {len(ngramas_seguridad)}")

# Cargar palabras de inseguridad y n-gramas de inseguridad
print(f"Leyendo palabras/expresiones de inseguridad de {ruta_inseguridad}")
palabras_inseguridad, ngramas_inseguridad = cargar_diccionario_con_ngramas(ruta_inseguridad)
print(f"Palabras de inseguridad cargadas: {len(palabras_inseguridad)}")
print(f"N-gramas de inseguridad cargados: {len(ngramas_inseguridad)}")

# Agregar después de las líneas de cargar diccionarios (línea ~77) y antes de la función analizar_archivo

# ==================== STOPLIST INTEGRADA ====================
# Palabras que NO deben considerarse para clasificación de polaridad/seguridad
stoplist = {
    'gobierno', 'morena', 'junior', 'dinero', 'politicos', 'grito', 'chile'
    'deberian', 'golfo', 'despues', 'jajajaja', 'jajaja', 'pagar', 
    'carlota', 'trump', 'pri', 'deberia', 'prian', 'propiedad', 'ano', 
    'rio', 'pagar renta', 'mama', 'alito', 'lucha', 'lady', 'pan', 
    'salinas', 'gringos', 'drones', 'kks', 'desfile', 'perro', 'andy', 
    'ninos', 'situacion', 'mar', 'investigacion', 'enchiladas', 'fox', 
    'regreso', 'qepd', 'virgen', 'pedir'
}

# Filtrar diccionarios para excluir stoplist
palabras_positivas = palabras_positivas - stoplist
palabras_negativas = palabras_negativas - stoplist
palabras_seguridad = palabras_seguridad - stoplist
palabras_inseguridad = palabras_inseguridad - stoplist

# Filtrar n-gramas que contengan palabras de stoplist
ngramas_positivos = {ng for ng in ngramas_positivos if not any(word in ng.split() for word in stoplist)}
ngramas_negativos = {ng for ng in ngramas_negativos if not any(word in ng.split() for word in stoplist)}
ngramas_seguridad = {ng for ng in ngramas_seguridad if not any(word in ng.split() for word in stoplist)}
ngramas_inseguridad = {ng for ng in ngramas_inseguridad if not any(word in ng.split() for word in stoplist)}

print(f"\n=== STOPLIST APLICADA ===")
print(f"Palabras en stoplist: {len(stoplist)}")
print(f"Palabras positivas después de filtrar: {len(palabras_positivas)}")
print(f"Palabras negativas después de filtrar: {len(palabras_negativas)}")
print(f"N-gramas positivos después de filtrar: {len(ngramas_positivos)}")
print(f"N-gramas negativos después de filtrar: {len(ngramas_negativos)}")
# ==================== FIN STOPLIST ====================

# Función para buscar n-gramas en el texto
def buscar_ngramas_en_texto(texto_limpio, ngramas):
    """
    Busca n-gramas en el texto y retorna las coincidencias encontradas
    """
    coincidencias = []
    texto_lower = texto_limpio.lower()
    
    for ngrama in ngramas:
        # Contar cuántas veces aparece el n-grama en el texto
        count = texto_lower.count(ngrama)
        if count > 0:
            coincidencias.extend([ngrama] * count)
    
    return coincidencias

# Función para extraer fecha del nombre de archivo
def extraer_fecha_archivo(nombre_archivo):
    # Buscar patrón de fecha YYYY-MM-DD
    match = re.search(r'(\d{4}-\d{2}-\d{2})', nombre_archivo)
    if match:
        return match.group(1)
    return None

# Función para analizar un archivo individual
def analizar_archivo(ruta_archivo):
    # Cargar texto del archivo
    texto_principal = leer_archivo(ruta_archivo)
    
    # Procesar el texto para palabras individuales
    palabras_texto = [p.strip().lower() for p in texto_principal.replace('.', ' ').replace(',', ' ').replace(';', ' ').replace(':', ' ').replace('!', ' ').replace('¡', ' ').replace('?', ' ').replace('¿', ' ').replace('-', ' ').replace('_', ' ').split() if p.strip()]
    
    # Texto limpio para búsqueda de n-gramas (mantener puntuación mínima)
    texto_limpio = texto_principal.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    # Normalizar espacios múltiples
    texto_limpio = ' '.join(texto_limpio.split())
    
    # Inicializar contadores para las palabras encontradas
    contador_positivas = 0
    contador_negativas = 0
    contador_seguridad = 0
    contador_inseguridad = 0
    
    # Listas para guardar las palabras/expresiones encontradas en cada categoría
    palabras_positivas_encontradas = []
    palabras_negativas_encontradas = []
    palabras_seguridad_encontradas = []
    palabras_inseguridad_encontradas = []
    
    # Analizar palabras individuales en el texto
    for palabra in palabras_texto:
        if palabra in palabras_positivas:
            contador_positivas += 1
            palabras_positivas_encontradas.append(palabra)
        if palabra in palabras_negativas:
            contador_negativas += 1
            palabras_negativas_encontradas.append(palabra)
        if palabra in palabras_seguridad:
            contador_seguridad += 1
            palabras_seguridad_encontradas.append(palabra)
        if palabra in palabras_inseguridad:
            contador_inseguridad += 1
            palabras_inseguridad_encontradas.append(palabra)
    
    # Buscar n-gramas en el texto
    ngramas_positivos_encontrados = buscar_ngramas_en_texto(texto_limpio, ngramas_positivos)
    ngramas_negativos_encontrados = buscar_ngramas_en_texto(texto_limpio, ngramas_negativos)
    ngramas_seguridad_encontrados = buscar_ngramas_en_texto(texto_limpio, ngramas_seguridad)
    ngramas_inseguridad_encontrados = buscar_ngramas_en_texto(texto_limpio, ngramas_inseguridad)
    
    # Agregar n-gramas a contadores y listas
    contador_positivas += len(ngramas_positivos_encontrados)
    contador_negativas += len(ngramas_negativos_encontrados)
    contador_seguridad += len(ngramas_seguridad_encontrados)
    contador_inseguridad += len(ngramas_inseguridad_encontrados)
    
    palabras_positivas_encontradas.extend(ngramas_positivos_encontrados)
    palabras_negativas_encontradas.extend(ngramas_negativos_encontrados)
    palabras_seguridad_encontradas.extend(ngramas_seguridad_encontrados)
    palabras_inseguridad_encontradas.extend(ngramas_inseguridad_encontrados)
    
    # Contar frecuencia de las palabras/expresiones en cada categoría
    contador_palabras_positivas = Counter(palabras_positivas_encontradas)
    contador_palabras_negativas = Counter(palabras_negativas_encontradas)
    contador_palabras_seguridad = Counter(palabras_seguridad_encontradas)
    contador_palabras_inseguridad = Counter(palabras_inseguridad_encontradas)
    
    # Obtener las 20 palabras/expresiones más frecuentes en cada categoría
    top_positivas = contador_palabras_positivas.most_common(20)
    top_negativas = contador_palabras_negativas.most_common(20)
    top_seguridad = contador_palabras_seguridad.most_common(20)
    top_inseguridad = contador_palabras_inseguridad.most_common(20)
    
    # Calcular porcentajes (polaridad)
    total_palabras_encontradas = contador_positivas + contador_negativas
    if total_palabras_encontradas > 0:
        porcentaje_positivas = (contador_positivas / total_palabras_encontradas) * 100
        porcentaje_negativas = (contador_negativas / total_palabras_encontradas) * 100
    else:
        porcentaje_positivas = 0
        porcentaje_negativas = 0
    
    # Calcular porcentajes (seguridad)
    total_palabras_seguridad_inseguridad = contador_seguridad + contador_inseguridad
    if total_palabras_seguridad_inseguridad > 0:
        porcentaje_seguridad = (contador_seguridad / total_palabras_seguridad_inseguridad) * 100
        porcentaje_inseguridad = (contador_inseguridad / total_palabras_seguridad_inseguridad) * 100
    else:
        porcentaje_seguridad = 0
        porcentaje_inseguridad = 0
    
    return {
        "contador_positivas": contador_positivas,
        "contador_negativas": contador_negativas,
        "total_palabras_encontradas": total_palabras_encontradas,
        "porcentaje_positivas": porcentaje_positivas,
        "porcentaje_negativas": porcentaje_negativas,
        "contador_seguridad": contador_seguridad,
        "contador_inseguridad": contador_inseguridad,
        "total_palabras_seguridad_inseguridad": total_palabras_seguridad_inseguridad,
        "porcentaje_seguridad": porcentaje_seguridad,
        "porcentaje_inseguridad": porcentaje_inseguridad,
        "top_positivas": top_positivas,
        "top_negativas": top_negativas,
        "top_seguridad": top_seguridad,
        "top_inseguridad": top_inseguridad,
        "total_palabras_texto": len(palabras_texto)
    }

# Almacenar todos los resultados
resultados_todos_directorios = []
resultados_por_directorio = {}

# Procesar cada carpeta
for carpeta in carpetas_analizar:
    print(f"\n=== PROCESANDO CARPETA: {carpeta} ===")
    ruta_carpeta = os.path.join(base_path, carpeta)
    
    if not os.path.exists(ruta_carpeta):
        print(f"Carpeta no encontrada: {ruta_carpeta}")
        continue
    
    # Obtener todos los archivos .txt de la carpeta
    archivos_txt = [f for f in os.listdir(ruta_carpeta) if f.endswith('.txt')]
    archivos_txt.sort()  # Ordenar por nombre
    
    resultados_carpeta = []
    
    for archivo in archivos_txt:
        print(f"Procesando: {archivo}")
        ruta_archivo = os.path.join(ruta_carpeta, archivo)
        
        # Extraer fecha del archivo
        fecha = extraer_fecha_archivo(archivo)
        if not fecha:
            print(f"No se pudo extraer fecha de {archivo}")
            continue
        
        # Analizar archivo
        resultado = analizar_archivo(ruta_archivo)
        resultado['archivo'] = archivo
        resultado['fecha'] = fecha
        resultado['carpeta'] = carpeta
        
        # Agregar terminación de fecha para línea de tiempo
        resultado['fecha_corta'] = fecha
        
        resultados_carpeta.append(resultado)
        resultados_todos_directorios.append(resultado)
    
    resultados_por_directorio[carpeta] = resultados_carpeta
    
    # Crear CSV para esta carpeta
    if resultados_carpeta:
        df_carpeta = pd.DataFrame(resultados_carpeta)
        csv_path = os.path.join(resultados_dir, f"{carpeta}_resultados_semanales.csv")
        df_carpeta.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"CSV guardado: {csv_path}")
        
        # Crear línea de tiempo para esta carpeta
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        fechas = [datetime.strptime(fecha, '%Y-%m-%d') for fecha in df_carpeta['fecha']]
        fechas_str = [fecha.strftime('%Y-%m-%d') for fecha in fechas]
        
        # Gráfico 1: Polaridad (Positivas vs Negativas)
        ax1.plot(fechas, df_carpeta['porcentaje_positivas'], marker='o', label='Positivas', color='#66b3ff')
        ax1.plot(fechas, df_carpeta['porcentaje_negativas'], marker='s', label='Negativas', color='#ff9999')
        ax1.set_title('Evolución de Polaridad (%)')
        ax1.set_ylabel('Porcentaje')
        ax1.legend()
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # Gráfico 2: Seguridad vs Inseguridad
        ax2.plot(fechas, df_carpeta['porcentaje_seguridad'], marker='o', label='Seguridad', color='#77dd77')
        ax2.plot(fechas, df_carpeta['porcentaje_inseguridad'], marker='s', label='Inseguridad', color='#ff6961')
        ax2.set_title('Evolución Seguridad/Inseguridad (%)')
        ax2.set_ylabel('Porcentaje')
        ax2.legend()
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3)
        
        # Gráfico 3: Conteo absoluto de palabras polaridad
        ax3.plot(fechas, df_carpeta['contador_positivas'], marker='o', label='Positivas', color='#66b3ff')
        ax3.plot(fechas, df_carpeta['contador_negativas'], marker='s', label='Negativas', color='#ff9999')
        ax3.set_title('Conteo Absoluto Polaridad')
        ax3.set_ylabel('Número de palabras')
        ax3.legend()
        ax3.tick_params(axis='x', rotation=45)
        ax3.grid(True, alpha=0.3)
        
        # Gráfico 4: Conteo absoluto seguridad
        ax4.plot(fechas, df_carpeta['contador_seguridad'], marker='o', label='Seguridad', color='#77dd77')
        ax4.plot(fechas, df_carpeta['contador_inseguridad'], marker='s', label='Inseguridad', color='#ff6961')
        ax4.set_title('Conteo Absoluto Seguridad/Inseguridad')
        ax4.set_ylabel('Número de palabras')
        ax4.legend()
        ax4.tick_params(axis='x', rotation=45)
        ax4.grid(True, alpha=0.3)
        
        plt.suptitle(f'Línea de Tiempo - {carpeta}', fontsize=16)
        plt.tight_layout()
        
        imagen_path = os.path.join(resultados_dir, f"{carpeta}_linea_tiempo.png")
        plt.savefig(imagen_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Línea de tiempo guardada: {imagen_path}")

# Crear CSV consolidado con todos los resultados
if resultados_todos_directorios:
    df_consolidado = pd.DataFrame(resultados_todos_directorios)
    csv_consolidado_path = os.path.join(resultados_dir, "resultados_consolidados_todas_carpetas.csv")
    df_consolidado.to_csv(csv_consolidado_path, index=False, encoding='utf-8')
    print(f"\nCSV consolidado guardado: {csv_consolidado_path}")
    
    # Crear línea de tiempo consolidada
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Agrupar por carpeta para diferentes líneas
    colores_carpetas = {'Facebook_Semanal_Txt': '#1f77b4', 'Fuentes_Integradas_Txt': '#ff7f0e', 
                       'X_Twitter_Semanal_Txt': '#2ca02c', 'Youtube_Semanal_Txt': '#d62728'}
    
    for carpeta in carpetas_analizar:
        df_carpeta = df_consolidado[df_consolidado['carpeta'] == carpeta]
        if not df_carpeta.empty:
            fechas = [datetime.strptime(fecha, '%Y-%m-%d') for fecha in df_carpeta['fecha']]
            
            # Gráfico 1: Polaridad positiva
            ax1.plot(fechas, df_carpeta['porcentaje_positivas'], 
                    marker='o', label=carpeta, color=colores_carpetas.get(carpeta, '#000000'))
            
            # Gráfico 2: Polaridad negativa  
            ax2.plot(fechas, df_carpeta['porcentaje_negativas'], 
                    marker='s', label=carpeta, color=colores_carpetas.get(carpeta, '#000000'))
            
            # Gráfico 3: Seguridad
            ax3.plot(fechas, df_carpeta['porcentaje_seguridad'], 
                    marker='^', label=carpeta, color=colores_carpetas.get(carpeta, '#000000'))
            
            # Gráfico 4: Inseguridad
            ax4.plot(fechas, df_carpeta['porcentaje_inseguridad'], 
                    marker='v', label=carpeta, color=colores_carpetas.get(carpeta, '#000000'))
    
    ax1.set_title('Evolución Polaridad Positiva (%)')
    ax1.set_ylabel('Porcentaje')
    ax1.legend()
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    
    ax2.set_title('Evolución Polaridad Negativa (%)')
    ax2.set_ylabel('Porcentaje')
    ax2.legend()
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    ax3.set_title('Evolución Seguridad (%)')
    ax3.set_ylabel('Porcentaje')
    ax3.legend()
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, alpha=0.3)
    
    ax4.set_title('Evolución Inseguridad (%)')
    ax4.set_ylabel('Porcentaje')
    ax4.legend()
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(True, alpha=0.3)
    
    plt.suptitle('Línea de Tiempo Consolidada - Todas las Carpetas', fontsize=16)
    plt.tight_layout()
    
    imagen_consolidada_path = os.path.join(resultados_dir, "linea_tiempo_consolidada_todas_carpetas.png")
    plt.savefig(imagen_consolidada_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Línea de tiempo consolidada guardada: {imagen_consolidada_path}")

# Crear informe de resumen
resumen_path = os.path.join(resultados_dir, "resumen_analisis.txt")
with open(resumen_path, 'w', encoding='utf-8') as f:
    f.write("=== RESUMEN DEL ANÁLISIS DE CLASIFICACIÓN SEMANAL ===\n\n")
    
    f.write("CARPETAS PROCESADAS:\n")
    for carpeta, resultados in resultados_por_directorio.items():
        f.write(f"- {carpeta}: {len(resultados)} archivos procesados\n")
    
    f.write(f"\nTOTAL DE ARCHIVOS PROCESADOS: {len(resultados_todos_directorios)}\n")
    
    f.write("\nARCHIVOS GENERADOS:\n")
    for carpeta in carpetas_analizar:
        f.write(f"- {carpeta}_resultados_semanales.csv\n")
        f.write(f"- {carpeta}_linea_tiempo.png\n")
    
    f.write("- resultados_consolidados_todas_carpetas.csv\n")
    f.write("- linea_tiempo_consolidada_todas_carpetas.png\n")
    f.write("- resumen_analisis.txt\n")
    
    f.write("\nNOTA: Se eliminó el filtrado de stoplist según solicitado.\n")
    f.write("Se implementó el soporte para n-gramas además de palabras individuales.\n")
    f.write("Todas las demás funciones del script original se mantuvieron intactas.\n")

print(f"\nResumen guardado: {resumen_path}")
print("\n=== ANÁLISIS COMPLETADO ===")
print(f"Todos los resultados se han guardado en: {resultados_dir}")

Leyendo palabras/expresiones positivas de /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios/diccionario_positivo_etiquetado.txt
Palabras positivas cargadas: 1810
N-gramas positivos cargados: 153
Leyendo palabras/expresiones negativas de /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios/diccionario_negativo_etiquetado.txt
Palabras negativas cargadas: 4040
N-gramas negativos cargados: 312
Leyendo palabras/expresiones de seguridad de /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios/diccionario_seguridad.txt
Palabras de seguridad cargadas: 195
N-gramas de seguridad cargados: 0
Leyendo palabras/expresiones de inseguridad de /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido

# Clasifficacion_Polaridad_Diccionarios_Estables

In [2]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import re
from datetime import datetime
import numpy as np

# Rutas base
base_path = "/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido"
resultados_dir = os.path.join(base_path, "Resultados_Clasificacion_Semanal_Etiquetado_Filtrado_Diccionarios_Estables")

# Carpetas a analizar
carpetas_analizar = [
    "Facebook_Semanal_Txt_Filtrado",
    "Fuentes_Integradas_Txt_Filtrado", 
    "X_Twitter_Semanal_Txt_Filtrado",
    "Youtube_Semanal_Txt_Filtrado"
]

# Rutas de diccionarios - usando los nuevos diccionarios con n-gramas
diccionarios_base = "/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios"
ruta_positivas = os.path.join(diccionarios_base, "diccionario_palabras_positivas.txt")
ruta_negativas = os.path.join(diccionarios_base, "diccionario_palabras_negativas.txt")
ruta_seguridad = os.path.join(diccionarios_base, "diccionario_seguridad.txt")
ruta_inseguridad = os.path.join(diccionarios_base, "diccionario_inseguridad.txt")

# Crear carpeta de resultados si no existe
if not os.path.exists(resultados_dir):
    os.makedirs(resultados_dir)
    print(f"Carpeta de resultados creada: {resultados_dir}")

# Función simple para leer un archivo de texto
def leer_archivo(ruta):
    try:
        with open(ruta, 'r', encoding='utf-8') as f:
            return f.read()
    except:
        try:
            with open(ruta, 'r', encoding='latin-1') as f:
                return f.read()
        except Exception as e:
            print(f"Error al leer {ruta}: {e}")
            return ""

# Función para separar palabras individuales de n-gramas
def cargar_diccionario_con_ngramas(ruta):
    texto = leer_archivo(ruta)
    palabras_individuales = set()
    ngramas = set()
    
    for linea in texto.splitlines():
        termino = linea.strip().lower()
        if termino:
            if ' ' in termino:  # Es un n-grama (tiene espacios)
                ngramas.add(termino)
            else:  # Es una palabra individual
                palabras_individuales.add(termino)
    
    return palabras_individuales, ngramas

# Cargar palabras positivas y n-gramas positivos
print(f"Leyendo palabras/expresiones positivas de {ruta_positivas}")
palabras_positivas, ngramas_positivos = cargar_diccionario_con_ngramas(ruta_positivas)
print(f"Palabras positivas cargadas: {len(palabras_positivas)}")
print(f"N-gramas positivos cargados: {len(ngramas_positivos)}")

# Cargar palabras negativas y n-gramas negativos
print(f"Leyendo palabras/expresiones negativas de {ruta_negativas}")
palabras_negativas, ngramas_negativos = cargar_diccionario_con_ngramas(ruta_negativas)
print(f"Palabras negativas cargadas: {len(palabras_negativas)}")
print(f"N-gramas negativos cargados: {len(ngramas_negativos)}")

# Cargar palabras de seguridad y n-gramas de seguridad
print(f"Leyendo palabras/expresiones de seguridad de {ruta_seguridad}")
palabras_seguridad, ngramas_seguridad = cargar_diccionario_con_ngramas(ruta_seguridad)
print(f"Palabras de seguridad cargadas: {len(palabras_seguridad)}")
print(f"N-gramas de seguridad cargados: {len(ngramas_seguridad)}")

# Cargar palabras de inseguridad y n-gramas de inseguridad
print(f"Leyendo palabras/expresiones de inseguridad de {ruta_inseguridad}")
palabras_inseguridad, ngramas_inseguridad = cargar_diccionario_con_ngramas(ruta_inseguridad)
print(f"Palabras de inseguridad cargadas: {len(palabras_inseguridad)}")
print(f"N-gramas de inseguridad cargados: {len(ngramas_inseguridad)}")

# Agregar después de las líneas de cargar diccionarios (línea ~77) y antes de la función analizar_archivo

# ==================== STOPLIST INTEGRADA ====================
# Palabras que NO deben considerarse para clasificación de polaridad/seguridad
stoplist = {
    'gobierno', 'morena', 'junior', 'dinero', 'politicos', 'grito', 'chile'
    'deberian', 'golfo', 'despues', 'jajajaja', 'jajaja', 'pagar', 
    'carlota', 'trump', 'pri', 'deberia', 'prian', 'propiedad', 'ano', 
    'rio', 'pagar renta', 'mama', 'alito', 'lucha', 'lady', 'pan', 
    'salinas', 'gringos', 'drones', 'kks', 'desfile', 'perro', 'andy', 
    'ninos', 'situacion', 'mar', 'investigacion', 'enchiladas', 'fox', 
    'regreso', 'qepd', 'virgen', 'transito', 'noche', 'policia', 'pedir'
}

# Filtrar diccionarios para excluir stoplist
palabras_positivas = palabras_positivas - stoplist
palabras_negativas = palabras_negativas - stoplist
palabras_seguridad = palabras_seguridad - stoplist
palabras_inseguridad = palabras_inseguridad - stoplist

# Filtrar n-gramas que contengan palabras de stoplist
ngramas_positivos = {ng for ng in ngramas_positivos if not any(word in ng.split() for word in stoplist)}
ngramas_negativos = {ng for ng in ngramas_negativos if not any(word in ng.split() for word in stoplist)}
ngramas_seguridad = {ng for ng in ngramas_seguridad if not any(word in ng.split() for word in stoplist)}
ngramas_inseguridad = {ng for ng in ngramas_inseguridad if not any(word in ng.split() for word in stoplist)}

print(f"\n=== STOPLIST APLICADA ===")
print(f"Palabras en stoplist: {len(stoplist)}")
print(f"Palabras positivas después de filtrar: {len(palabras_positivas)}")
print(f"Palabras negativas después de filtrar: {len(palabras_negativas)}")
print(f"N-gramas positivos después de filtrar: {len(ngramas_positivos)}")
print(f"N-gramas negativos después de filtrar: {len(ngramas_negativos)}")
# ==================== FIN STOPLIST ====================

# Función para buscar n-gramas en el texto
def buscar_ngramas_en_texto(texto_limpio, ngramas):
    """
    Busca n-gramas en el texto y retorna las coincidencias encontradas
    """
    coincidencias = []
    texto_lower = texto_limpio.lower()
    
    for ngrama in ngramas:
        # Contar cuántas veces aparece el n-grama en el texto
        count = texto_lower.count(ngrama)
        if count > 0:
            coincidencias.extend([ngrama] * count)
    
    return coincidencias

# Función para extraer fecha del nombre de archivo
def extraer_fecha_archivo(nombre_archivo):
    # Buscar patrón de fecha YYYY-MM-DD
    match = re.search(r'(\d{4}-\d{2}-\d{2})', nombre_archivo)
    if match:
        return match.group(1)
    return None

# Función para analizar un archivo individual
def analizar_archivo(ruta_archivo):
    # Cargar texto del archivo
    texto_principal = leer_archivo(ruta_archivo)
    
    # Procesar el texto para palabras individuales
    palabras_texto = [p.strip().lower() for p in texto_principal.replace('.', ' ').replace(',', ' ').replace(';', ' ').replace(':', ' ').replace('!', ' ').replace('¡', ' ').replace('?', ' ').replace('¿', ' ').replace('-', ' ').replace('_', ' ').split() if p.strip()]
    
    # Texto limpio para búsqueda de n-gramas (mantener puntuación mínima)
    texto_limpio = texto_principal.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    # Normalizar espacios múltiples
    texto_limpio = ' '.join(texto_limpio.split())
    
    # Inicializar contadores para las palabras encontradas
    contador_positivas = 0
    contador_negativas = 0
    contador_seguridad = 0
    contador_inseguridad = 0
    
    # Listas para guardar las palabras/expresiones encontradas en cada categoría
    palabras_positivas_encontradas = []
    palabras_negativas_encontradas = []
    palabras_seguridad_encontradas = []
    palabras_inseguridad_encontradas = []
    
    # Analizar palabras individuales en el texto
    for palabra in palabras_texto:
        if palabra in palabras_positivas:
            contador_positivas += 1
            palabras_positivas_encontradas.append(palabra)
        if palabra in palabras_negativas:
            contador_negativas += 1
            palabras_negativas_encontradas.append(palabra)
        if palabra in palabras_seguridad:
            contador_seguridad += 1
            palabras_seguridad_encontradas.append(palabra)
        if palabra in palabras_inseguridad:
            contador_inseguridad += 1
            palabras_inseguridad_encontradas.append(palabra)
    
    # Buscar n-gramas en el texto
    ngramas_positivos_encontrados = buscar_ngramas_en_texto(texto_limpio, ngramas_positivos)
    ngramas_negativos_encontrados = buscar_ngramas_en_texto(texto_limpio, ngramas_negativos)
    ngramas_seguridad_encontrados = buscar_ngramas_en_texto(texto_limpio, ngramas_seguridad)
    ngramas_inseguridad_encontrados = buscar_ngramas_en_texto(texto_limpio, ngramas_inseguridad)
    
    # Agregar n-gramas a contadores y listas
    contador_positivas += len(ngramas_positivos_encontrados)
    contador_negativas += len(ngramas_negativos_encontrados)
    contador_seguridad += len(ngramas_seguridad_encontrados)
    contador_inseguridad += len(ngramas_inseguridad_encontrados)
    
    palabras_positivas_encontradas.extend(ngramas_positivos_encontrados)
    palabras_negativas_encontradas.extend(ngramas_negativos_encontrados)
    palabras_seguridad_encontradas.extend(ngramas_seguridad_encontrados)
    palabras_inseguridad_encontradas.extend(ngramas_inseguridad_encontrados)
    
    # Contar frecuencia de las palabras/expresiones en cada categoría
    contador_palabras_positivas = Counter(palabras_positivas_encontradas)
    contador_palabras_negativas = Counter(palabras_negativas_encontradas)
    contador_palabras_seguridad = Counter(palabras_seguridad_encontradas)
    contador_palabras_inseguridad = Counter(palabras_inseguridad_encontradas)
    
    # Obtener las 20 palabras/expresiones más frecuentes en cada categoría
    top_positivas = contador_palabras_positivas.most_common(20)
    top_negativas = contador_palabras_negativas.most_common(20)
    top_seguridad = contador_palabras_seguridad.most_common(20)
    top_inseguridad = contador_palabras_inseguridad.most_common(20)
    
    # Calcular porcentajes (polaridad)
    total_palabras_encontradas = contador_positivas + contador_negativas
    if total_palabras_encontradas > 0:
        porcentaje_positivas = (contador_positivas / total_palabras_encontradas) * 100
        porcentaje_negativas = (contador_negativas / total_palabras_encontradas) * 100
    else:
        porcentaje_positivas = 0
        porcentaje_negativas = 0
    
    # Calcular porcentajes (seguridad)
    total_palabras_seguridad_inseguridad = contador_seguridad + contador_inseguridad
    if total_palabras_seguridad_inseguridad > 0:
        porcentaje_seguridad = (contador_seguridad / total_palabras_seguridad_inseguridad) * 100
        porcentaje_inseguridad = (contador_inseguridad / total_palabras_seguridad_inseguridad) * 100
    else:
        porcentaje_seguridad = 0
        porcentaje_inseguridad = 0
    
    return {
        "contador_positivas": contador_positivas,
        "contador_negativas": contador_negativas,
        "total_palabras_encontradas": total_palabras_encontradas,
        "porcentaje_positivas": porcentaje_positivas,
        "porcentaje_negativas": porcentaje_negativas,
        "contador_seguridad": contador_seguridad,
        "contador_inseguridad": contador_inseguridad,
        "total_palabras_seguridad_inseguridad": total_palabras_seguridad_inseguridad,
        "porcentaje_seguridad": porcentaje_seguridad,
        "porcentaje_inseguridad": porcentaje_inseguridad,
        "top_positivas": top_positivas,
        "top_negativas": top_negativas,
        "top_seguridad": top_seguridad,
        "top_inseguridad": top_inseguridad,
        "total_palabras_texto": len(palabras_texto)
    }

# Almacenar todos los resultados
resultados_todos_directorios = []
resultados_por_directorio = {}

# Procesar cada carpeta
for carpeta in carpetas_analizar:
    print(f"\n=== PROCESANDO CARPETA: {carpeta} ===")
    ruta_carpeta = os.path.join(base_path, carpeta)
    
    if not os.path.exists(ruta_carpeta):
        print(f"Carpeta no encontrada: {ruta_carpeta}")
        continue
    
    # Obtener todos los archivos .txt de la carpeta
    archivos_txt = [f for f in os.listdir(ruta_carpeta) if f.endswith('.txt')]
    archivos_txt.sort()  # Ordenar por nombre
    
    resultados_carpeta = []
    
    for archivo in archivos_txt:
        print(f"Procesando: {archivo}")
        ruta_archivo = os.path.join(ruta_carpeta, archivo)
        
        # Extraer fecha del archivo
        fecha = extraer_fecha_archivo(archivo)
        if not fecha:
            print(f"No se pudo extraer fecha de {archivo}")
            continue
        
        # Analizar archivo
        resultado = analizar_archivo(ruta_archivo)
        resultado['archivo'] = archivo
        resultado['fecha'] = fecha
        resultado['carpeta'] = carpeta
        
        # Agregar terminación de fecha para línea de tiempo
        resultado['fecha_corta'] = fecha
        
        resultados_carpeta.append(resultado)
        resultados_todos_directorios.append(resultado)
    
    resultados_por_directorio[carpeta] = resultados_carpeta
    
    # Crear CSV para esta carpeta
    if resultados_carpeta:
        df_carpeta = pd.DataFrame(resultados_carpeta)
        csv_path = os.path.join(resultados_dir, f"{carpeta}_resultados_semanales.csv")
        df_carpeta.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"CSV guardado: {csv_path}")
        
        # Crear línea de tiempo para esta carpeta
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        fechas = [datetime.strptime(fecha, '%Y-%m-%d') for fecha in df_carpeta['fecha']]
        fechas_str = [fecha.strftime('%Y-%m-%d') for fecha in fechas]
        
        # Gráfico 1: Polaridad (Positivas vs Negativas)
        ax1.plot(fechas, df_carpeta['porcentaje_positivas'], marker='o', label='Positivas', color='#66b3ff')
        ax1.plot(fechas, df_carpeta['porcentaje_negativas'], marker='s', label='Negativas', color='#ff9999')
        ax1.set_title('Evolución de Polaridad (%)')
        ax1.set_ylabel('Porcentaje')
        ax1.legend()
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # Gráfico 2: Seguridad vs Inseguridad
        ax2.plot(fechas, df_carpeta['porcentaje_seguridad'], marker='o', label='Seguridad', color='#77dd77')
        ax2.plot(fechas, df_carpeta['porcentaje_inseguridad'], marker='s', label='Inseguridad', color='#ff6961')
        ax2.set_title('Evolución Seguridad/Inseguridad (%)')
        ax2.set_ylabel('Porcentaje')
        ax2.legend()
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3)
        
        # Gráfico 3: Conteo absoluto de palabras polaridad
        ax3.plot(fechas, df_carpeta['contador_positivas'], marker='o', label='Positivas', color='#66b3ff')
        ax3.plot(fechas, df_carpeta['contador_negativas'], marker='s', label='Negativas', color='#ff9999')
        ax3.set_title('Conteo Absoluto Polaridad')
        ax3.set_ylabel('Número de palabras')
        ax3.legend()
        ax3.tick_params(axis='x', rotation=45)
        ax3.grid(True, alpha=0.3)
        
        # Gráfico 4: Conteo absoluto seguridad
        ax4.plot(fechas, df_carpeta['contador_seguridad'], marker='o', label='Seguridad', color='#77dd77')
        ax4.plot(fechas, df_carpeta['contador_inseguridad'], marker='s', label='Inseguridad', color='#ff6961')
        ax4.set_title('Conteo Absoluto Seguridad/Inseguridad')
        ax4.set_ylabel('Número de palabras')
        ax4.legend()
        ax4.tick_params(axis='x', rotation=45)
        ax4.grid(True, alpha=0.3)
        
        plt.suptitle(f'Línea de Tiempo - {carpeta}', fontsize=16)
        plt.tight_layout()
        
        imagen_path = os.path.join(resultados_dir, f"{carpeta}_linea_tiempo.png")
        plt.savefig(imagen_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Línea de tiempo guardada: {imagen_path}")

# Crear CSV consolidado con todos los resultados
if resultados_todos_directorios:
    df_consolidado = pd.DataFrame(resultados_todos_directorios)
    csv_consolidado_path = os.path.join(resultados_dir, "resultados_consolidados_todas_carpetas.csv")
    df_consolidado.to_csv(csv_consolidado_path, index=False, encoding='utf-8')
    print(f"\nCSV consolidado guardado: {csv_consolidado_path}")
    
    # Crear línea de tiempo consolidada
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Agrupar por carpeta para diferentes líneas
    colores_carpetas = {'Facebook_Semanal_Txt': '#1f77b4', 'Fuentes_Integradas_Txt': '#ff7f0e', 
                       'X_Twitter_Semanal_Txt': '#2ca02c', 'Youtube_Semanal_Txt': '#d62728'}
    
    for carpeta in carpetas_analizar:
        df_carpeta = df_consolidado[df_consolidado['carpeta'] == carpeta]
        if not df_carpeta.empty:
            fechas = [datetime.strptime(fecha, '%Y-%m-%d') for fecha in df_carpeta['fecha']]
            
            # Gráfico 1: Polaridad positiva
            ax1.plot(fechas, df_carpeta['porcentaje_positivas'], 
                    marker='o', label=carpeta, color=colores_carpetas.get(carpeta, '#000000'))
            
            # Gráfico 2: Polaridad negativa  
            ax2.plot(fechas, df_carpeta['porcentaje_negativas'], 
                    marker='s', label=carpeta, color=colores_carpetas.get(carpeta, '#000000'))
            
            # Gráfico 3: Seguridad
            ax3.plot(fechas, df_carpeta['porcentaje_seguridad'], 
                    marker='^', label=carpeta, color=colores_carpetas.get(carpeta, '#000000'))
            
            # Gráfico 4: Inseguridad
            ax4.plot(fechas, df_carpeta['porcentaje_inseguridad'], 
                    marker='v', label=carpeta, color=colores_carpetas.get(carpeta, '#000000'))
    
    ax1.set_title('Evolución Polaridad Positiva (%)')
    ax1.set_ylabel('Porcentaje')
    ax1.legend()
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    
    ax2.set_title('Evolución Polaridad Negativa (%)')
    ax2.set_ylabel('Porcentaje')
    ax2.legend()
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    ax3.set_title('Evolución Seguridad (%)')
    ax3.set_ylabel('Porcentaje')
    ax3.legend()
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, alpha=0.3)
    
    ax4.set_title('Evolución Inseguridad (%)')
    ax4.set_ylabel('Porcentaje')
    ax4.legend()
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(True, alpha=0.3)
    
    plt.suptitle('Línea de Tiempo Consolidada - Todas las Carpetas', fontsize=16)
    plt.tight_layout()
    
    imagen_consolidada_path = os.path.join(resultados_dir, "linea_tiempo_consolidada_todas_carpetas.png")
    plt.savefig(imagen_consolidada_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Línea de tiempo consolidada guardada: {imagen_consolidada_path}")

# Crear informe de resumen
resumen_path = os.path.join(resultados_dir, "resumen_analisis.txt")
with open(resumen_path, 'w', encoding='utf-8') as f:
    f.write("=== RESUMEN DEL ANÁLISIS DE CLASIFICACIÓN SEMANAL ===\n\n")
    
    f.write("CARPETAS PROCESADAS:\n")
    for carpeta, resultados in resultados_por_directorio.items():
        f.write(f"- {carpeta}: {len(resultados)} archivos procesados\n")
    
    f.write(f"\nTOTAL DE ARCHIVOS PROCESADOS: {len(resultados_todos_directorios)}\n")
    
    f.write("\nARCHIVOS GENERADOS:\n")
    for carpeta in carpetas_analizar:
        f.write(f"- {carpeta}_resultados_semanales.csv\n")
        f.write(f"- {carpeta}_linea_tiempo.png\n")
    
    f.write("- resultados_consolidados_todas_carpetas.csv\n")
    f.write("- linea_tiempo_consolidada_todas_carpetas.png\n")
    f.write("- resumen_analisis.txt\n")
    
    f.write("\nNOTA: Se eliminó el filtrado de stoplist según solicitado.\n")
    f.write("Se implementó el soporte para n-gramas además de palabras individuales.\n")
    f.write("Todas las demás funciones del script original se mantuvieron intactas.\n")

print(f"\nResumen guardado: {resumen_path}")
print("\n=== ANÁLISIS COMPLETADO ===")
print(f"Todos los resultados se han guardado en: {resultados_dir}")

Leyendo palabras/expresiones positivas de /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios/diccionario_palabras_positivas.txt
Palabras positivas cargadas: 1528
N-gramas positivos cargados: 0
Leyendo palabras/expresiones negativas de /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios/diccionario_palabras_negativas.txt
Palabras negativas cargadas: 3166
N-gramas negativos cargados: 0
Leyendo palabras/expresiones de seguridad de /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Diccionarios/diccionario_seguridad.txt
Palabras de seguridad cargadas: 195
N-gramas de seguridad cargados: 0
Leyendo palabras/expresiones de inseguridad de /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido/Dicci

# Clasificacion_Temas

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import re
import logging
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# Configuración de logging simple para Jupyter
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def validar_rutas_y_archivos() -> Tuple[Path, Path, Path]:
    """Valida que todas las rutas necesarias existan"""
    ROOT_DIR = Path("/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido")
    
    # Ruta EXACTA del diccionario según las indicaciones del usuario
    DICCIONARIO_PATH = Path("/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Diccionario/Diccionario_Temas_V5.csv")
    
    # Verificar que el diccionario existe
    if not DICCIONARIO_PATH.exists():
        raise FileNotFoundError(f"Diccionario no encontrado en: {DICCIONARIO_PATH}")
    
    # Verificar que el directorio de corpus existe  
    if not ROOT_DIR.exists():
        raise FileNotFoundError(f"Directorio de corpus no encontrado: {ROOT_DIR}")
    
    logging.info(f"✅ Diccionario encontrado: {DICCIONARIO_PATH}")
    logging.info(f"✅ Directorio de corpus: {ROOT_DIR}")
    
    SALIDA_CSV = ROOT_DIR / "Resultados_Temas_Semanales_Mejorado_V5.csv"
    
    return ROOT_DIR, DICCIONARIO_PATH, SALIDA_CSV

def cargar_diccionario_temas(diccionario_path: Path) -> Dict[str, Dict[str, List[str]]]:
    """Carga y procesa el diccionario de temas"""
    logging.info(f"Cargando diccionario desde: {diccionario_path}")
    
    try:
        df_dic = pd.read_csv(diccionario_path)
    except Exception as e:
        logging.error(f"Error al cargar diccionario: {e}")
        raise
    
    temas = {}
    for col in df_dic.columns:
        terms = [
            str(x).strip().lower()
            for x in df_dic[col].dropna().astype(str)
            if str(x).strip() and str(x).strip().lower() != 'nan'
        ]
        
        # Separar unigramas y n-gramas
        unigrams = [t for t in terms if " " not in t]
        ngrams = [t for t in terms if " " in t]
        
        temas[col] = {"unigrams": unigrams, "ngrams": ngrams}
        logging.info(f"Tema '{col}': {len(unigrams)} unigramas, {len(ngrams)} n-gramas")
    
    return temas

def cargar_o_crear_stoplists(diccionario_path: Path) -> Dict[str, List[str]]:
    """Carga o crea archivo de stoplists específicas por tema"""
    
    stoplist_path = diccionario_path.parent / "Stoplists_Por_Tema_1.csv"
    
    # Definir stoplists por defecto con solo "uno" en cada tema
    stoplists_default = {
        "Corrupcion": ["uno"],
        "Inseguridad": ["uno"],
        "Basura": ["uno"],
        "Alumbrado": ["uno"],
        "Agua": ["uno"],
        "Vialidades": ["uno"],
        "Presencia_MV": ["uno"],
        "Bienestar": ["uno"],
        "Morena": ["uno"],
        "Economia": ["uno"]
    }
    
    if stoplist_path.exists():
        try:
            df_stoplist = pd.read_csv(stoplist_path)
            stoplists = {}
            
            for col in df_stoplist.columns:
                stopwords = [
                    str(x).strip().lower()
                    for x in df_stoplist[col].dropna().astype(str)
                    if str(x).strip() and str(x).strip().lower() != 'nan'
                ]
                stoplists[col] = stopwords
            
            logging.info(f"✅ Stoplists cargadas desde: {stoplist_path}")
            
            # Mostrar resumen de stoplists
            for tema, stopwords in stoplists.items():
                if stopwords:
                    logging.info(f"  🚫 {tema}: {len(stopwords)} palabras bloqueadas")
            
            return stoplists
            
        except Exception as e:
            logging.warning(f"Error cargando stoplists: {e}, usando defaults")
    
    # Si no existe, crear archivo con stoplists por defecto
    logging.info("📝 Creando archivo de stoplists por defecto...")
    
    # Encontrar la longitud máxima para crear DataFrame
    max_len = max(len(words) for words in stoplists_default.values()) if stoplists_default else 0
    
    # Rellenar listas cortas con valores vacíos
    df_data = {}
    for tema, stopwords in stoplists_default.items():
        padded_words = stopwords + [''] * (max_len - len(stopwords))
        df_data[tema] = padded_words
    
    df_stoplist = pd.DataFrame(df_data)
    df_stoplist.to_csv(stoplist_path, index=False, encoding='utf-8')
    
    logging.info(f"✅ Archivo de stoplists creado: {stoplist_path}")
    logging.info("📋 PUEDES EDITARLO para personalizar las palabras bloqueadas por tema")
    logging.info("💡 Después de editar, vuelve a ejecutar el script")
    
    return stoplists_default

def aplicar_stoplists(terminos: List[str], stoplist: List[str]) -> List[str]:
    """Aplica stoplist a una lista de términos"""
    if not stoplist:
        return terminos
    
    terminos_filtrados = []
    for termino in terminos:
        # Verificar si el término completo está en la stoplist
        if termino not in stoplist:
            # Para n-gramas, verificar si contiene palabras de la stoplist
            if " " in termino:
                palabras_termino = termino.split()
                contiene_stop = any(palabra in stoplist for palabra in palabras_termino)
                if not contiene_stop:
                    terminos_filtrados.append(termino)
            else:
                terminos_filtrados.append(termino)
    
    return terminos_filtrados

def extraer_fecha_archivo(filename: str) -> str:
    """Extrae fecha del nombre de archivo con múltiples patrones"""
    # Múltiples patrones de fecha posibles
    fecha_patterns = [
        r"\d{4}-\d{2}-\d{2}",  # 2024-10-15
        r"\d{4}_\d{2}_\d{2}",  # 2024_10_15
        r"\d{8}",              # 20241015
        r"\d{4}-W\d{2}",       # 2024-W42
    ]
    
    for pattern in fecha_patterns:
        match = re.search(pattern, filename)
        if match:
            return match.group(0)
    
    # Si no encuentra patrón de fecha, usar el nombre completo sin extensión
    logging.warning(f"No se encontró patrón de fecha en: {filename}")
    return Path(filename).stem

def contar_tema_en_texto(texto: str, tema_info: Dict[str, List[str]], palabra_tracker: Dict[str, Dict[str, int]], tema_actual: str) -> Tuple[int, bool]:
    """
    Cuenta apariciones de un tema en texto y determina presencia binaria
    NOTA: tema_info ya debe venir con stoplists aplicadas
    
    Args:
        texto: texto a analizar
        tema_info: diccionario con unigramas y ngramas del tema (ya filtrados)
        palabra_tracker: diccionario para trackear palabras específicas
        tema_actual: nombre del tema actual
    
    Returns:
        Tuple[int, bool]: (conteo_total, tema_presente)
    """
    tokens = texto.split()
    token_counts = Counter(tokens)
    padded_text = " " + texto + " "
    
    conteo_total = 0
    tema_presente = False
    
    # Contar unigramas (ya filtrados por stoplist)
    for term in tema_info["unigrams"]:
        count = token_counts.get(term, 0)
        if count > 0:
            conteo_total += count
            tema_presente = True
            # Trackear palabra específica
            palabra_tracker[tema_actual][term] += count
    
    # Contar n-gramas (ya filtrados por stoplist)
    for ngram in tema_info["ngrams"]:
        term_padded = " " + ngram + " "
        start = 0
        count_ngram = 0
        
        while True:
            idx = padded_text.find(term_padded, start)
            if idx == -1:
                break
            count_ngram += 1
            start = idx + 1
        
        if count_ngram > 0:
            conteo_total += count_ngram
            tema_presente = True
            # Trackear palabra específica
            palabra_tracker[tema_actual][ngram] += count_ngram
    
    return conteo_total, tema_presente

def analizar_archivo_mejorado(path_txt: Path, temas_dict: Dict[str, Dict[str, List[str]]], palabra_tracker: Dict[str, Dict[str, int]]) -> pd.DataFrame:
    """Analiza un archivo semanal con métricas mejoradas"""
    
    if not path_txt.exists():
        logging.error(f"Archivo no encontrado: {path_txt}")
        return pd.DataFrame()
    
    total_comentarios = 0
    total_palabras = 0
    
    # Inicializar estadísticas para cada tema
    stats = {
        tema: {
            "tokens_totales": 0,          # Conteo total de menciones
            "comentarios_con_tema": 0,     # Comentarios que mencionan el tema (binario)
        } 
        for tema in temas_dict
    }
    
    try:
        with path_txt.open(encoding="utf-8", errors="replace") as f:
            for line_num, raw_line in enumerate(f, 1):
                line = raw_line.strip()
                if not line:
                    continue
                
                total_comentarios += 1
                texto = line.lower()
                tokens = texto.split()
                total_palabras += len(tokens)
                
                # Analizar cada tema en el comentario actual
                for tema, tema_info in temas_dict.items():
                    conteo_tema, tema_presente = contar_tema_en_texto(texto, tema_info, palabra_tracker, tema)
                    
                    stats[tema]["tokens_totales"] += conteo_tema
                    if tema_presente:
                        stats[tema]["comentarios_con_tema"] += 1
                        
    except Exception as e:
        logging.error(f"Error procesando archivo {path_txt}: {e}")
        return pd.DataFrame()
    
    # Construir DataFrame de resultados
    filas = []
    for tema, s in stats.items():
        tokens_totales = s["tokens_totales"]
        comentarios_con_tema = s["comentarios_con_tema"]
        
        # Métricas normalizadas
        intensidad_por_1000 = (tokens_totales / total_palabras * 1000) if total_palabras > 0 else 0.0
        presencia_tema_pct = (comentarios_con_tema / total_comentarios * 100) if total_comentarios > 0 else 0.0
        
        filas.append({
            "tema": tema,
            "total_comentarios": total_comentarios,
            "total_palabras": total_palabras,
            "tokens_totales": tokens_totales,
            "intensidad_por_1000": round(intensidad_por_1000, 3),
            "comentarios_con_tema": comentarios_con_tema,
            "presencia_tema_pct": round(presencia_tema_pct, 2),
        })
    
    resultado = pd.DataFrame(filas)
    logging.info(f"Procesado {path_txt.name}: {total_comentarios} comentarios, {total_palabras} palabras")
    
    return resultado

def convertir_fecha_string(fecha_str: str) -> datetime:
    """Convierte string de fecha a objeto datetime"""
    # Intentar múltiples formatos de fecha
    formatos = ["%Y-%m-%d", "%Y_%m_%d", "%Y%m%d", "%Y-W%W"]
    
    for formato in formatos:
        try:
            return datetime.strptime(fecha_str, formato)
        except ValueError:
            continue
    
    # Si no funciona ningún formato, usar fecha dummy
    logging.warning(f"No se pudo convertir fecha: {fecha_str}, usando fecha dummy")
    return datetime(2024, 1, 1)

def suavizar_series_temporales(df: pd.DataFrame, ventana: int = 3) -> pd.DataFrame:
    """
    Aplica suavizado a las series temporales por red y tema usando numpy
    
    Args:
        df: DataFrame con resultados
        ventana: tamaño de ventana para suavizado (default: 3 semanas)
    """
    df_suavizado = df.copy()
    
    # Asegurar que semana sea datetime
    df_suavizado['fecha'] = df_suavizado['semana'].apply(convertir_fecha_string)
    
    # Suavizar por red y tema
    for red in df_suavizado['red'].unique():
        for tema in df_suavizado['tema'].unique():
            mask = (df_suavizado['red'] == red) & (df_suavizado['tema'] == tema)
            subset = df_suavizado[mask].sort_values('fecha').copy()
            
            if len(subset) >= ventana:
                # Aplicar suavizado manual con numpy
                for columna in ['intensidad_por_1000', 'presencia_tema_pct']:
                    if columna in subset.columns:
                        valores = subset[columna].values
                        valores_suavizados = np.convolve(valores, np.ones(ventana)/ventana, mode='same')
                        df_suavizado.loc[mask & df_suavizado['fecha'].isin(subset['fecha']), f'{columna}_suavizado'] = valores_suavizados
            else:
                # Si hay pocos datos, usar valores originales
                for columna in ['intensidad_por_1000', 'presencia_tema_pct']:
                    if columna in subset.columns:
                        df_suavizado.loc[mask, f'{columna}_suavizado'] = subset[columna].values
    
    return df_suavizado

def generar_graficas_seguimiento(df: pd.DataFrame, salida_dir: Path):
    """Genera gráficas de seguimiento temporal por tema y red"""
    
    # Crear directorio para gráficas
    graficas_dir = salida_dir / "Graficas_Seguimiento"
    graficas_dir.mkdir(exist_ok=True)
    
    # Configurar estilo básico
    plt.style.use('default')
    
    # Colores para diferentes redes
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    # Asegurar que hay columna de fecha
    if 'fecha' not in df.columns:
        df['fecha'] = df['semana'].apply(convertir_fecha_string)
    
    df_sorted = df.sort_values(['red', 'tema', 'fecha'])
    
    # 1. Gráfica por tema (todas las redes juntas)
    temas = df['tema'].unique()
    
    for tema in temas:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        tema_data = df_sorted[df_sorted['tema'] == tema]
        
        # Intensidad por 1000
        for i, red in enumerate(tema_data['red'].unique()):
            red_data = tema_data[tema_data['red'] == red]
            color = colors[i % len(colors)]
            
            # Original
            ax1.plot(red_data['fecha'], red_data['intensidad_por_1000'], 
                    'o-', alpha=0.6, label=f'{red} (original)', markersize=4, color=color)
            
            # Suavizado (si existe)
            if 'intensidad_por_1000_suavizado' in red_data.columns:
                ax1.plot(red_data['fecha'], red_data['intensidad_por_1000_suavizado'], 
                        '-', linewidth=2, label=f'{red} (suavizado)', color=color)
        
        ax1.set_title(f'{tema} - Intensidad por 1000 palabras')
        ax1.set_ylabel('Intensidad por 1000')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Presencia tema %
        for i, red in enumerate(tema_data['red'].unique()):
            red_data = tema_data[tema_data['red'] == red]
            color = colors[i % len(colors)]
            
            # Original
            ax2.plot(red_data['fecha'], red_data['presencia_tema_pct'], 
                    'o-', alpha=0.6, label=f'{red} (original)', markersize=4, color=color)
            
            # Suavizado (si existe)
            if 'presencia_tema_pct_suavizado' in red_data.columns:
                ax2.plot(red_data['fecha'], red_data['presencia_tema_pct_suavizado'], 
                        '-', linewidth=2, label=f'{red} (suavizado)', color=color)
        
        ax2.set_title(f'{tema} - % Comentarios que mencionan el tema')
        ax2.set_ylabel('% Comentarios')
        ax2.set_xlabel('Fecha')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Formatear fechas en eje X
        for ax in [ax1, ax2]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
            ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
        
        plt.tight_layout()
        plt.savefig(graficas_dir / f'{tema}_seguimiento.png', dpi=300, bbox_inches='tight')
        plt.close()
    
    # 2. Gráfica comparativa de todos los temas (una red a la vez)
    for red in df['red'].unique():
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))
        
        red_data = df_sorted[df_sorted['red'] == red]
        
        for i, tema in enumerate(red_data['tema'].unique()):
            tema_data = red_data[red_data['tema'] == tema]
            color = colors[i % len(colors)]
            
            # Usar suavizado si está disponible, sino original
            intensidad_col = 'intensidad_por_1000_suavizado' if 'intensidad_por_1000_suavizado' in tema_data.columns else 'intensidad_por_1000'
            presencia_col = 'presencia_tema_pct_suavizado' if 'presencia_tema_pct_suavizado' in tema_data.columns else 'presencia_tema_pct'
            
            ax1.plot(tema_data['fecha'], tema_data[intensidad_col], 
                    'o-', label=tema, markersize=4, color=color)
            ax2.plot(tema_data['fecha'], tema_data[presencia_col], 
                    'o-', label=tema, markersize=4, color=color)
        
        ax1.set_title(f'{red} - Intensidad por tema (por 1000 palabras)')
        ax1.set_ylabel('Intensidad por 1000')
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.grid(True, alpha=0.3)
        
        ax2.set_title(f'{red} - Presencia por tema (% comentarios)')
        ax2.set_ylabel('% Comentarios')
        ax2.set_xlabel('Fecha')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.grid(True, alpha=0.3)
        
        # Formatear fechas
        for ax in [ax1, ax2]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
            ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
        
        plt.tight_layout()
        plt.savefig(graficas_dir / f'{red}_todos_temas.png', dpi=300, bbox_inches='tight')
        plt.close()
    
    logging.info(f"Gráficas guardadas en: {graficas_dir}")

def generar_reporte_palabras_frecuentes(palabra_tracker: Dict[str, Dict[str, int]], salida_dir: Path, top_n: int = 25):
    """Genera reporte de las palabras más frecuentes por tema"""
    
    reporte_path = salida_dir / "Palabras_Mas_Frecuentes_Top25.xlsx"
    
    with pd.ExcelWriter(reporte_path, engine='openpyxl') as writer:
        resumen_general = []
        
        for tema, palabras_count in palabra_tracker.items():
            if not palabras_count:  # Si está vacío
                continue
                
            # Ordenar palabras por frecuencia
            palabras_ordenadas = sorted(palabras_count.items(), key=lambda x: x[1], reverse=True)
            top_palabras = palabras_ordenadas[:top_n]
            
            # Crear DataFrame para este tema
            df_tema = pd.DataFrame(top_palabras, columns=['Palabra', 'Frecuencia'])
            df_tema['Tema'] = tema
            df_tema['Ranking'] = range(1, len(df_tema) + 1)
            df_tema['Porcentaje'] = (df_tema['Frecuencia'] / df_tema['Frecuencia'].sum() * 100).round(2)
            
            # Reordenar columnas
            df_tema = df_tema[['Ranking', 'Palabra', 'Frecuencia', 'Porcentaje', 'Tema']]
            
            # Guardar en hoja separada
            sheet_name = tema[:30]  # Excel limita nombres de hoja
            df_tema.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Agregar al resumen general
            if top_palabras:
                resumen_general.extend([
                    {
                        'Tema': tema,
                        'Total_Palabras_Diferentes': len(palabras_count),
                        'Total_Menciones': sum(palabras_count.values()),
                        'Palabra_Mas_Frecuente': top_palabras[0][0],
                        'Frecuencia_Top1': top_palabras[0][1],
                        'Top5_Palabras': ', '.join([p[0] for p in top_palabras[:5]])
                    }
                ])
        
        # Crear resumen general
        if resumen_general:
            df_resumen = pd.DataFrame(resumen_general)
            df_resumen = df_resumen.sort_values('Total_Menciones', ascending=False)
            df_resumen.to_excel(writer, sheet_name='RESUMEN_GENERAL', index=False)
    
    logging.info(f"Reporte de palabras frecuentes guardado en: {reporte_path}")

def main():
    """Función principal"""
    try:
        # 1. Validar rutas y cargar diccionario
        ROOT_DIR, DICCIONARIO_PATH, SALIDA_CSV = validar_rutas_y_archivos()
        temas_originales = cargar_diccionario_temas(DICCIONARIO_PATH)
        
        # 2. Cargar stoplists específicas por tema
        stoplists = cargar_o_crear_stoplists(DICCIONARIO_PATH)
        
        # 3. Aplicar stoplists a cada tema
        temas_filtrados = {}
        logging.info("🔄 Aplicando stoplists por tema...")
        
        for tema, tema_data in temas_originales.items():
            stoplist_tema = stoplists.get(tema, [])
            
            # Aplicar stoplist a unigramas y n-gramas
            unigrams_filtrados = aplicar_stoplists(tema_data["unigrams"], stoplist_tema)
            ngrams_filtrados = aplicar_stoplists(tema_data["ngrams"], stoplist_tema)
            
            temas_filtrados[tema] = {
                "unigrams": unigrams_filtrados,
                "ngrams": ngrams_filtrados
            }
            
            # Reportar filtrado
            original_count = len(tema_data["unigrams"]) + len(tema_data["ngrams"])
            filtered_count = len(unigrams_filtrados) + len(ngrams_filtrados)
            blocked_count = original_count - filtered_count
            
            if blocked_count > 0:
                logging.info(f"  🚫 {tema}: {blocked_count} términos bloqueados, {filtered_count} términos activos")
            else:
                logging.info(f"  ✅ {tema}: {filtered_count} términos activos (sin filtrado)")
        
        # 4. Inicializar tracker de palabras
        palabra_tracker = defaultdict(lambda: defaultdict(int))
        
        # 5. Buscar carpetas de redes sociales
        carpetas_encontradas = []
        # Patrones actualizados basados en los nombres reales de las carpetas
        patrones_carpetas = [
            "_Semanal_Txt_Filtrado", 
            "_Semanal_Txt", 
            "_Integradas_Txt_Filtrado",
            "_Twitter_Semanal_Txt_Filtrado"
        ]
        
        # También buscar carpetas específicas por nombre exacto
        nombres_exactos = [
            "Youtube_Semanal_Txt_Filtrado",
            "Facebook_Semanal_Txt_Filtrado", 
            "Fuentes_Integradas_Txt_Filtrado",
            "X_Twitter_Semanal_Txt_Filtrado",
            "Youtube_Semanal_Txt",
            "X_Twitter_Semanal_Txt",
            "Fuentes_Integradas_Txt",
            "Facebook_Semanal_Txt"
        ]
        
        for carpeta in ROOT_DIR.iterdir():
            if not carpeta.is_dir():
                continue
            
            # Verificar si coincide con algún patrón o nombre exacto
            coincide = (
                any(patron in carpeta.name for patron in patrones_carpetas) or
                carpeta.name in nombres_exactos
            )
            
            # Excluir carpetas de resultados y diccionarios
            if coincide and "Resultado" not in carpeta.name and "Diccionario" not in carpeta.name:
                carpetas_encontradas.append(carpeta)
        
        if not carpetas_encontradas:
            logging.warning(f"No se encontraron carpetas de redes en: {ROOT_DIR}")
            logging.info("Carpetas disponibles:")
            for item in ROOT_DIR.iterdir():
                if item.is_dir():
                    logging.info(f"  - {item.name}")
            return
        
        logging.info(f"Carpetas de redes encontradas: {len(carpetas_encontradas)}")
        
        # 6. Procesar cada carpeta y archivo
        resultados = []
        archivos_procesados = 0
        
        for carpeta in carpetas_encontradas:
            # Extraer nombre de red con mejor lógica
            if "Youtube" in carpeta.name:
                red = "Youtube"
            elif "Facebook" in carpeta.name:
                red = "Facebook"  
            elif "Twitter" in carpeta.name or "X_Twitter" in carpeta.name:
                red = "X_Twitter"
            elif "Fuentes" in carpeta.name:
                red = "Fuentes_Integradas"
            else:
                red = carpeta.name.split("_")[0]  # Fallback
                
            logging.info(f"Procesando carpeta: {carpeta.name} (red = {red})")
            
            archivos_txt = list(carpeta.glob("*.txt"))
            if not archivos_txt:
                logging.warning(f"No se encontraron archivos .txt en {carpeta.name}")
                continue
            
            logging.info(f"  Encontrados {len(archivos_txt)} archivos .txt")
            
            for path_txt in sorted(archivos_txt):
                semana = extraer_fecha_archivo(path_txt.name)
                
                # Usar temas_filtrados en lugar de temas
                df_semana = analizar_archivo_mejorado(path_txt, temas_filtrados, palabra_tracker)
                
                if not df_semana.empty:
                    df_semana["red"] = red
                    df_semana["archivo"] = path_txt.name
                    df_semana["semana"] = semana
                    
                    resultados.append(df_semana)
                    archivos_procesados += 1
        
        # 7. Consolidar resultados
        if resultados:
            df_final = pd.concat(resultados, ignore_index=True)
            
            # Reordenar columnas para claridad
            columnas_ordenadas = [
                "red", "semana", "archivo", "tema",
                "total_comentarios", "total_palabras",
                "tokens_totales", "intensidad_por_1000",
                "comentarios_con_tema", "presencia_tema_pct"
            ]
            
            df_final = df_final[columnas_ordenadas]
            
            # 8. Generar versión suavizada
            logging.info("Generando series suavizadas...")
            df_suavizado = suavizar_series_temporales(df_final, ventana=3)
            
            # 9. Crear directorio de resultados
            resultados_dir = ROOT_DIR / "Resultados_Analisis_Tematico"
            resultados_dir.mkdir(exist_ok=True)
            
            # 10. Exportar CSVs
            # Archivo original
            archivo_original = resultados_dir / "Resultados_Temas_Semanales_Original.csv"
            df_final.to_csv(archivo_original, index=False, encoding='utf-8')
            
            # Archivo suavizado
            archivo_suavizado = resultados_dir / "Resultados_Temas_Semanales_Suavizado.csv"
            df_suavizado.to_csv(archivo_suavizado, index=False, encoding='utf-8')
            
            # 11. Generar gráficas de seguimiento
            logging.info("Generando gráficas de seguimiento...")
            generar_graficas_seguimiento(df_suavizado, resultados_dir)
            
            # 12. Generar reporte de palabras frecuentes
            logging.info("Generando reporte de palabras más frecuentes...")
            generar_reporte_palabras_frecuentes(dict(palabra_tracker), resultados_dir, top_n=25)
            
            # 13. Estadísticas finales
            redes_procesadas = df_final['red'].nunique()
            semanas_procesadas = df_final['semana'].nunique()
            temas_analizados = df_final['tema'].nunique()
            
            logging.info("="*60)
            logging.info("PROCESO COMPLETADO EXITOSAMENTE")
            logging.info("="*60)
            logging.info(f"Directorio de resultados: {resultados_dir}")
            logging.info(f"Archivos procesados: {archivos_procesados}")
            logging.info(f"Redes procesadas: {redes_procesadas}")
            logging.info(f"Semanas procesadas: {semanas_procesadas}")
            logging.info(f"Temas analizados: {temas_analizados}")
            logging.info(f"Total de filas en resultado: {len(df_final)}")
            
            # 14. Archivos generados
            logging.info("\n📁 ARCHIVOS GENERADOS:")
            logging.info(f"  📊 {archivo_original.name}")
            logging.info(f"  📈 {archivo_suavizado.name}")
            logging.info(f"  📋 Palabras_Mas_Frecuentes_Top25.xlsx")
            logging.info(f"  🚫 Stoplists_Por_Tema.csv (editable)")
            logging.info(f"  📂 Graficas_Seguimiento/ (gráficas por tema y red)")
            
            # 15. Mostrar resumen por red
            logging.info("\n📊 RESUMEN POR RED SOCIAL:")
            resumen_red = df_final.groupby('red').agg({
                'semana': 'nunique',
                'total_comentarios': 'sum',
                'total_palabras': 'sum'
            }).round(0)
            
            for red, datos in resumen_red.iterrows():
                logging.info(f"  🔹 {red}: {datos['semana']} semanas, {datos['total_comentarios']:,.0f} comentarios, {datos['total_palabras']:,.0f} palabras")
            
            # 16. Mostrar top palabras por tema (preview)
            logging.info("\n🔤 TOP 3 PALABRAS MÁS FRECUENTES POR TEMA:")
            for tema, palabras_count in palabra_tracker.items():
                if palabras_count:
                    top_3 = sorted(palabras_count.items(), key=lambda x: x[1], reverse=True)[:3]
                    palabras_str = ", ".join([f"{palabra}({freq})" for palabra, freq in top_3])
                    logging.info(f"  📝 {tema}: {palabras_str}")
            
            # 17. Información sobre stoplists
            logging.info("\n🚫 STOPLISTS APLICADAS:")
            for tema, stoplist in stoplists.items():
                if stoplist:
                    logging.info(f"  ❌ {tema}: {', '.join(stoplist[:5])}{'...' if len(stoplist) > 5 else ''}")
            
            logging.info(f"\n💡 Para personalizar stoplists, edita: {DICCIONARIO_PATH.parent / 'Stoplists_Por_Tema.csv'}")
            logging.info("💡 Luego vuelve a ejecutar el script para aplicar los cambios")
            
        else:
            logging.error("No se generaron resultados. Revisa las rutas y contenido de archivos.")
            
    except Exception as e:
        logging.error(f"Error en el proceso principal: {e}")
        raise

if __name__ == "__main__":
    main()

INFO: ✅ Diccionario encontrado: /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Diccionario/Diccionario_Temas_V5.csv
INFO: ✅ Directorio de corpus: /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Corpus_Clasificacion_Dividido
INFO: Cargando diccionario desde: /home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Diccionario/Diccionario_Temas_V5.csv
INFO: Tema 'Corrupcion': 77 unigramas, 1 n-gramas
INFO: Tema 'Inseguridad': 78 unigramas, 12 n-gramas
INFO: Tema 'Basura': 42 unigramas, 3 n-gramas
INFO: Tema 'Alumbrado': 41 unigramas, 28 n-gramas
INFO: Tema 'Agua': 42 unigramas, 11 n-gramas
INFO: Tema 'Vialidades': 60 unigramas, 3 n-gramas
INFO: Tema 'Presencia_MV': 2 unigramas, 20 n-gramas
INFO: Tema 'Bienestar': 28 unigramas, 26 n-gramas
INFO: Tema 'Morena': 25 unigramas, 24 n-gramas
INFO: Tema 'Economia': 47 unigramas, 36 n-gramas
INFO: 📝 Creando archivo de stoplists

# Datos_Modelo_Entero 

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import logging

# Configuración de logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def fecha_a_semana_iso(fecha):
    """Convierte fecha a código de semana ISO (YYYY-WXX)"""
    if pd.isna(fecha):
        return None
    
    if isinstance(fecha, str):
        fecha_obj = datetime.strptime(fecha[:10], '%Y-%m-%d')
    else:
        fecha_obj = fecha
    
    year, week, _ = fecha_obj.isocalendar()
    return f'{year}-W{week:02d}'

def limpiar_columnas_sentimientos(df):
    """Limpia y estandariza nombres de columnas de sentimientos"""
    # Mapeo de nombres de redes
    red_mapping = {
        'Fuentes_Juntas': 'FuentesIntegradas',
        'Facebook': 'Facebook', 
        'Twitter': 'XTwitter',
        'Youtube': 'Youtube'
    }
    
    columnas_limpias = []
    for col in df.columns:
        if col.startswith('Positivo_'):
            # Extraer componentes del nombre
            parts = col.split('_')
            red = parts[1] if len(parts) > 1 else 'Unknown'
            
            # Estandarizar nombre de red
            red_std = red_mapping.get(red, red)
            
            # Determinar tipo de métrica
            if 'MA5' in col:
                tipo = 'MA5'
            else:
                tipo = 'Original'
            
            # Determinar diccionario
            if 'Etiquetado' in col:
                dic = 'Etiquetado'
            else:
                dic = 'Original'
            
            nuevo_nombre = f'Sentimiento_{red_std}_{tipo}_{dic}'
            columnas_limpias.append(nuevo_nombre)
        else:
            columnas_limpias.append(col)
    
    return columnas_limpias

def cargar_y_procesar_datos():
    """Carga y procesa todos los datasets"""
    
    datasets = {}
    
    logging.info("🔄 Cargando datasets...")
    
    # Ruta base donde están los archivos
    BASE_PATH = Path("/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Datos_Modelo_1")
    
    # 1. Análisis Temático
    try:
        df_temas_orig = pd.read_csv(BASE_PATH / 'Resultados_Temas_Semanales_Original.csv')
        df_temas_suav = pd.read_csv(BASE_PATH / 'Resultados_Temas_Semanales_Suavizado.csv')
        
        # Agregar semana ISO
        df_temas_orig['semana_iso'] = df_temas_orig['semana'].apply(fecha_a_semana_iso)
        df_temas_suav['semana_iso'] = df_temas_suav['semana'].apply(fecha_a_semana_iso)
        
        datasets['temas_original'] = df_temas_orig
        datasets['temas_suavizado'] = df_temas_suav
        
        logging.info(f"✅ Temático: {len(df_temas_orig)} filas, {df_temas_orig['semana_iso'].nunique()} semanas")
        
    except Exception as e:
        logging.error(f"❌ Error cargando temático: {e}")
        return None
    
    # 2. Análisis de Sentimientos
    try:
        df_sent = pd.read_excel(BASE_PATH / 'Resultados_Semanales_Dos_Diccionarios_Suavizados.xlsx')
        
        # Limpiar nombres de columnas
        columnas_nuevas = limpiar_columnas_sentimientos(df_sent)
        df_sent.columns = columnas_nuevas
        
        # Agregar semana ISO
        df_sent['semana_iso'] = df_sent.iloc[:, 0].apply(fecha_a_semana_iso)
        
        datasets['sentimientos'] = df_sent
        
        logging.info(f"✅ Sentimientos: {len(df_sent)} filas, {df_sent['semana_iso'].nunique()} semanas")
        
    except Exception as e:
        logging.error(f"❌ Error cargando sentimientos: {e}")
        return None
    
    # 3. Métricas de Facebook
    try:
        df_fb = pd.read_excel(BASE_PATH / 'Facebook_Metricas_Semanal.xlsx')
        
        # Agregar semana ISO
        df_fb['semana_iso'] = df_fb['Fecha_Semanal'].apply(fecha_a_semana_iso)
        
        # Renombrar columnas con prefijo
        columnas_fb = ['semana_iso'] + [f'FB_{col}' for col in df_fb.columns if col not in ['Fecha_Semanal', 'semana_iso']]
        df_fb_clean = df_fb[['semana_iso'] + [col for col in df_fb.columns if col not in ['Fecha_Semanal', 'semana_iso']]].copy()
        df_fb_clean.columns = columnas_fb
        
        datasets['facebook'] = df_fb_clean
        
        logging.info(f"✅ Facebook: {len(df_fb_clean)} filas, {df_fb_clean['semana_iso'].nunique()} semanas")
        
    except Exception as e:
        logging.error(f"❌ Error cargando Facebook: {e}")
        datasets['facebook'] = None
    
    # 4. Métricas de Twitter
    try:
        df_tw = pd.read_excel(BASE_PATH / 'Twitter_Metricas_Semanal.xlsx')
        
        # Agregar semana ISO
        df_tw['semana_iso'] = df_tw['Fecha_Semanal'].apply(fecha_a_semana_iso)
        
        # Renombrar columnas con prefijo
        columnas_tw = ['semana_iso'] + [f'TW_{col}' for col in df_tw.columns if col not in ['Fecha_Semanal', 'semana_iso']]
        df_tw_clean = df_tw[['semana_iso'] + [col for col in df_tw.columns if col not in ['Fecha_Semanal', 'semana_iso']]].copy()
        df_tw_clean.columns = columnas_tw
        
        datasets['twitter'] = df_tw_clean
        
        logging.info(f"✅ Twitter: {len(df_tw_clean)} filas, {df_tw_clean['semana_iso'].nunique()} semanas")
        
    except Exception as e:
        logging.error(f"⚠️ Error cargando Twitter: {e}")
        datasets['twitter'] = None
    
    # 5. Sala de Prensa
    try:
        df_sala = pd.read_csv(BASE_PATH / 'SalaPrensa_TODO_2024-10_a_2025-11_semanal.csv')
        
        # Agregar semana ISO
        df_sala['semana_iso'] = df_sala['lunes_semana'].apply(fecha_a_semana_iso)
        
        # Renombrar columnas con prefijo
        columnas_sala = ['semana_iso'] + [f'Agenda_{col}' for col in df_sala.columns if col not in ['lunes_semana', 'anio_iso', 'semana_iso', 'domingo_semana']]
        df_sala_clean = df_sala[['semana_iso'] + [col for col in df_sala.columns if col not in ['lunes_semana', 'anio_iso', 'semana_iso', 'domingo_semana']]].copy()
        df_sala_clean.columns = columnas_sala
        
        datasets['agenda'] = df_sala_clean
        
        logging.info(f"✅ Agenda: {len(df_sala_clean)} filas, {df_sala_clean['semana_iso'].nunique()} semanas")
        
    except Exception as e:
        logging.error(f"❌ Error cargando Agenda: {e}")
        return None
    
    return datasets

def transformar_temas_formato_ancho(df_temas, sufijo=""):
    """Transforma datos temáticos de formato largo a ancho"""
    
    # Variables a pivotar
    variables_interes = ['intensidad_por_1000', 'presencia_tema_pct']
    if sufijo == "_suavizado":
        variables_interes = ['intensidad_por_1000_suavizado', 'presencia_tema_pct_suavizado']
    
    resultado = None
    
    for variable in variables_interes:
        if variable in df_temas.columns:
            # Crear tabla pivoteada
            pivot = df_temas.pivot_table(
                index='semana_iso',
                columns=['red', 'tema'],
                values=variable,
                aggfunc='first'
            )
            
            # Aplanar nombres de columnas
            pivot.columns = [f'{variable}_{red}_{tema}{sufijo}' for red, tema in pivot.columns]
            
            if resultado is None:
                resultado = pivot
            else:
                resultado = pd.concat([resultado, pivot], axis=1)
    
    return resultado.reset_index()

def integrar_datasets(datasets):
    """Integra todos los datasets en un DataFrame final"""
    
    logging.info("🔄 Integrando datasets...")
    
    # Encontrar semanas comunes
    semanas_comunes = set(datasets['temas_original']['semana_iso'].unique())
    
    for nombre, df in datasets.items():
        if df is not None and 'semana_iso' in df.columns:
            semanas_dataset = set(df['semana_iso'].unique())
            semanas_comunes = semanas_comunes.intersection(semanas_dataset)
            logging.info(f"  📊 {nombre}: {len(semanas_dataset)} semanas")
    
    logging.info(f"🎯 Semanas en común: {len(semanas_comunes)}")
    
    # Crear DataFrame base con semanas comunes
    df_final = pd.DataFrame({'semana_iso': sorted(list(semanas_comunes))})
    
    # 1. Integrar Sentimientos (ya está en formato ancho)
    df_sent = datasets['sentimientos']
    df_sent_filtrado = df_sent[df_sent['semana_iso'].isin(semanas_comunes)]
    
    # Eliminar primera columna (fecha original) y quedarnos con todo menos semana_iso para merge
    sent_cols = [col for col in df_sent_filtrado.columns if col not in [df_sent_filtrado.columns[0]]]
    df_sent_merge = df_sent_filtrado[sent_cols]
    
    df_final = df_final.merge(df_sent_merge, on='semana_iso', how='left')
    
    logging.info(f"✅ Sentimientos integrados: {len(df_sent_merge.columns)-1} variables")
    
    # 2. Integrar Temas Originales (formato ancho)
    df_temas_orig_filtrado = datasets['temas_original'][datasets['temas_original']['semana_iso'].isin(semanas_comunes)]
    df_temas_ancho = transformar_temas_formato_ancho(df_temas_orig_filtrado, "")
    
    df_final = df_final.merge(df_temas_ancho, on='semana_iso', how='left')
    
    logging.info(f"✅ Temas originales integrados: {len(df_temas_ancho.columns)-1} variables")
    
    # 3. Integrar Temas Suavizados (formato ancho)
    df_temas_suav_filtrado = datasets['temas_suavizado'][datasets['temas_suavizado']['semana_iso'].isin(semanas_comunes)]
    
    # Buscar columnas suavizadas
    columnas_suavizadas = [col for col in df_temas_suav_filtrado.columns if '_suavizado' in col]
    
    if len(columnas_suavizadas) > 0:
        df_temas_suav_ancho = transformar_temas_formato_ancho(df_temas_suav_filtrado, "_suavizado")
        df_final = df_final.merge(df_temas_suav_ancho, on='semana_iso', how='left')
        logging.info(f"✅ Temas suavizados integrados: {len(df_temas_suav_ancho.columns)-1} variables")
    else:
        logging.info("⚠️ No se encontraron columnas suavizadas en temas")
    
    # 4. Integrar Facebook
    if datasets['facebook'] is not None:
        df_fb_filtrado = datasets['facebook'][datasets['facebook']['semana_iso'].isin(semanas_comunes)]
        df_final = df_final.merge(df_fb_filtrado, on='semana_iso', how='left')
        logging.info(f"✅ Facebook integrado: {len(df_fb_filtrado.columns)-1} variables")
    
    # 5. Integrar Twitter
    if datasets['twitter'] is not None:
        df_tw_filtrado = datasets['twitter'][datasets['twitter']['semana_iso'].isin(semanas_comunes)]
        df_final = df_final.merge(df_tw_filtrado, on='semana_iso', how='left')
        logging.info(f"✅ Twitter integrado: {len(df_tw_filtrado.columns)-1} variables")
    
    # 6. Integrar Agenda
    df_agenda_filtrado = datasets['agenda'][datasets['agenda']['semana_iso'].isin(semanas_comunes)]
    df_final = df_final.merge(df_agenda_filtrado, on='semana_iso', how='left')
    
    logging.info(f"✅ Agenda integrada: {len(df_agenda_filtrado.columns)-1} variables")
    
    return df_final

def generar_variables_modelo(df_final):
    """Genera variables específicas para modelado"""
    
    df_modelo = df_final.copy()
    
    logging.info("🔄 Generando variables de modelo...")
    
    # 1. Variables dependientes (Y) - Sentimientos principales
    variables_y = {
        'Y_Sentimiento_FuentesIntegradas_MA5_Original': 'Sentimiento_FuentesIntegradas_MA5_Original',
        'Y_Sentimiento_Facebook_MA5_Original': 'Sentimiento_Facebook_MA5_Original', 
        'Y_Sentimiento_XTwitter_MA5_Original': 'Sentimiento_XTwitter_MA5_Original',
        'Y_Sentimiento_Youtube_MA5_Original': 'Sentimiento_Youtube_MA5_Original'
    }
    
    for nuevo_nombre, col_original in variables_y.items():
        if col_original in df_modelo.columns:
            df_modelo[nuevo_nombre] = df_modelo[col_original]
    
    # 2. Variables de agenda agregadas
    columnas_agenda = [col for col in df_modelo.columns if col.startswith('Agenda_')]
    
    if len(columnas_agenda) > 0:
        # Total de comunicados por semana
        df_modelo['X_Agenda_Total'] = df_modelo[[col for col in columnas_agenda if 'total' in col.lower()]].sum(axis=1)
        
        # Intensidad de agenda por categoría
        categorias_agenda = ['mv', 'populares', 'inversion', 'trabajo_empleo', 'seguridad', 'corrupcion']
        for cat in categorias_agenda:
            col_cat = f'Agenda_notas_{cat}'
            if col_cat in df_modelo.columns:
                df_modelo[f'X_Agenda_{cat.capitalize()}'] = df_modelo[col_cat]
    
    # 3. Variables temáticas agregadas por red
    redes = ['Facebook', 'XTwitter', 'Youtube', 'FuentesIntegradas']
    temas = ['Corrupcion', 'Inseguridad', 'Basura', 'Alumbrado', 'Agua', 'Vialidades', 'Presencia_MV', 'Bienestar', 'Morena', 'Economia']
    
    for red in redes:
        for tema in temas:
            # Intensidad suavizada (preferible para modelo)
            col_intensidad_suav = f'intensidad_por_1000_suavizado_{red}_{tema}_suavizado'
            col_presencia_suav = f'presencia_tema_pct_suavizado_{red}_{tema}_suavizado'
            
            # Si no existe suavizada, usar original
            col_intensidad_orig = f'intensidad_por_1000_{red}_{tema}'
            col_presencia_orig = f'presencia_tema_pct_{red}_{tema}'
            
            # Variables de intensidad
            if col_intensidad_suav in df_modelo.columns:
                df_modelo[f'X_Intensidad_{red}_{tema}'] = df_modelo[col_intensidad_suav]
            elif col_intensidad_orig in df_modelo.columns:
                df_modelo[f'X_Intensidad_{red}_{tema}'] = df_modelo[col_intensidad_orig]
            
            # Variables de presencia
            if col_presencia_suav in df_modelo.columns:
                df_modelo[f'X_Presencia_{red}_{tema}'] = df_modelo[col_presencia_suav]
            elif col_presencia_orig in df_modelo.columns:
                df_modelo[f'X_Presencia_{red}_{tema}'] = df_modelo[col_presencia_orig]
    
    # 4. Variables de engagement
    if any(col.startswith('FB_') for col in df_modelo.columns):
        # Facebook engagement
        df_modelo['X_FB_Engagement_Total'] = df_modelo.get('FB_Total_Engagement', 0)
        df_modelo['X_FB_Engagement_por_Post'] = df_modelo.get('FB_Engagement_por_Post', 0)
        df_modelo['X_FB_Posts_Totales'] = df_modelo.get('FB_Total_Posts', 0)
    
    if any(col.startswith('TW_') for col in df_modelo.columns):
        # Twitter engagement
        df_modelo['X_TW_Engagement_Total'] = df_modelo.get('TW_Total_Engagement', 0)
        df_modelo['X_TW_Engagement_por_Post'] = df_modelo.get('TW_Engagement_por_Post', 0)
        df_modelo['X_TW_Posts_Totales'] = df_modelo.get('TW_Total_Posts', 0)
    
    # 5. Agregar fecha legible
    df_modelo['Fecha_Semana'] = df_modelo['semana_iso'].apply(
        lambda x: f"{x.split('-W')[0]}-Semana{x.split('-W')[1]}" if x and '-W' in str(x) else x
    )
    
    logging.info(f"✅ Variables de modelo generadas")
    logging.info(f"📊 Dataset final: {len(df_modelo)} filas × {len(df_modelo.columns)} columnas")
    
    return df_modelo

def generar_resumen_variables(df_modelo):
    """Genera resumen de variables disponibles para modelado"""
    
    variables_y = [col for col in df_modelo.columns if col.startswith('Y_')]
    variables_x_agenda = [col for col in df_modelo.columns if col.startswith('X_Agenda_')]
    variables_x_intensidad = [col for col in df_modelo.columns if col.startswith('X_Intensidad_')]
    variables_x_presencia = [col for col in df_modelo.columns if col.startswith('X_Presencia_')]
    variables_x_engagement = [col for col in df_modelo.columns if col.startswith('X_FB_') or col.startswith('X_TW_')]
    
    resumen = {
        'Variables_Dependientes_Y': variables_y,
        'Variables_Agenda_X': variables_x_agenda,
        'Variables_Intensidad_Temas_X': variables_x_intensidad[:10],  # Solo primeras 10
        'Variables_Presencia_Temas_X': variables_x_presencia[:10],   # Solo primeras 10  
        'Variables_Engagement_X': variables_x_engagement,
        'Total_Variables_Y': len(variables_y),
        'Total_Variables_X': len(variables_x_agenda) + len(variables_x_intensidad) + len(variables_x_presencia) + len(variables_x_engagement),
        'Total_Observaciones': len(df_modelo),
        'Rango_Fechas': f"{df_modelo['semana_iso'].min()} a {df_modelo['semana_iso'].max()}"
    }
    
    return resumen

def main():
    """Función principal de integración"""
    
    logging.info("🚀 INICIANDO INTEGRACIÓN DE DATASETS")
    logging.info("="*50)
    
    # 1. Cargar todos los datos
    datasets = cargar_y_procesar_datos()
    if datasets is None:
        logging.error("❌ Error crítico cargando datos")
        return
    
    # 2. Integrar en dataset único
    df_final = integrar_datasets(datasets)
    
    # 3. Generar variables de modelo
    df_modelo = generar_variables_modelo(df_final)
    
    # 4. Generar resumen
    resumen = generar_resumen_variables(df_modelo)
    
    # 5. Exportar resultados
    output_dir = Path('/home/emilio/Documentos/Lab/tampico_env/Proyectos/IPSEL_Tampico/Ml_Monica_Villarreal/Datos/Datos_Modelo_1')
    
    # Dataset completo
    archivo_completo = output_dir / 'Dataset_Integrado_Completo_1.csv'
    df_final.to_csv(archivo_completo, index=False, encoding='utf-8')
    
    # Dataset para modelo
    archivo_modelo = output_dir / 'Dataset_Modelo_Variables_XY_1.csv'
    df_modelo.to_csv(archivo_modelo, index=False, encoding='utf-8')
    
    # Resumen de variables
    archivo_resumen = output_dir / 'Resumen_Variables_Modelo_1.txt'
    with open(archivo_resumen, 'w', encoding='utf-8') as f:
        f.write("RESUMEN DE VARIABLES PARA MODELADO\n")
        f.write("="*50 + "\n\n")
        
        for categoria, variables in resumen.items():
            if isinstance(variables, list):
                f.write(f"{categoria} ({len(variables)}):\n")
                for var in variables[:15]:  # Máximo 15 por categoría
                    f.write(f"  - {var}\n")
                if len(variables) > 15:
                    f.write(f"  ... y {len(variables)-15} más\n")
                f.write("\n")
            else:
                f.write(f"{categoria}: {variables}\n")
    
    # 6. Mostrar resumen final
    logging.info("="*50)
    logging.info("🎉 INTEGRACIÓN COMPLETADA EXITOSAMENTE")
    logging.info("="*50)
    logging.info(f"📊 Dataset integrado: {len(df_final)} semanas × {len(df_final.columns)} variables totales")
    logging.info(f"🎯 Dataset modelo: {len(df_modelo)} semanas × {len(df_modelo.columns)} variables")
    logging.info(f"📈 Variables Y (dependientes): {resumen['Total_Variables_Y']}")
    logging.info(f"📉 Variables X (independientes): {resumen['Total_Variables_X']}")
    logging.info(f"📅 Período: {resumen['Rango_Fechas']}")
    
    logging.info(f"\n📁 ARCHIVOS GENERADOS:")
    logging.info(f"  📊 {archivo_completo.name} - Dataset completo integrado")
    logging.info(f"  🎯 {archivo_modelo.name} - Variables preparadas para modelo")
    logging.info(f"  📋 {archivo_resumen.name} - Resumen de variables")
    
    logging.info("\n💡 PRÓXIMOS PASOS:")
    logging.info("  1. Revisar variables en el archivo de resumen")
    logging.info("  2. Explorar correlaciones entre X e Y")
    logging.info("  3. Construir modelos de regresión/predicción")
    logging.info("  4. Validar hipótesis sobre agenda → sentimiento")
    
    return df_modelo, resumen

if __name__ == "__main__":
    resultado = main()

INFO: 🚀 INICIANDO INTEGRACIÓN DE DATASETS
INFO: ==================================================
INFO: 🔄 Cargando datasets...
INFO: ✅ Temático: 4400 filas, 55 semanas
INFO: ✅ Sentimientos: 55 filas, 55 semanas
INFO: ✅ Facebook: 59 filas, 59 semanas
INFO: ✅ Twitter: 55 filas, 55 semanas
INFO: ✅ Agenda: 59 filas, 59 semanas
INFO: 🔄 Integrando datasets...
INFO:   📊 temas_original: 55 semanas
INFO:   📊 temas_suavizado: 55 semanas
INFO:   📊 sentimientos: 55 semanas
INFO:   📊 facebook: 59 semanas
INFO:   📊 twitter: 55 semanas
INFO:   📊 agenda: 59 semanas
INFO: 🎯 Semanas en común: 55
INFO: ✅ Sentimientos integrados: 17 variables
INFO: ✅ Temas originales integrados: 80 variables
INFO: ✅ Temas suavizados integrados: 80 variables
INFO: ✅ Facebook integrado: 13 variables
INFO: ✅ Twitter integrado: 14 variables
INFO: ✅ Agenda integrada: 7 variables
INFO: 🔄 Generando variables de modelo...
INFO: ✅ Variables de modelo generadas
INFO: 📊 Dataset final: 55 filas × 269 columnas
INFO: =================